In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import src.hypothesis_tests as ht

In [2]:
df = pd.read_csv("../data/raw/insurance_data.csv")

print(df.head())
print(df.shape)

  CustomerID  Age  Gender     Province VehicleType  AnnualIncome  RiskScore  \
0  AC-100000   56    Male  Addis Ababa       Sedan        147270         61   
1  AC-100001   69  Female  Addis Ababa         SUV         74640         57   
2  AC-100002   46    Male       Oromia       Sedan         70555         42   
3  AC-100003   32  Female       Somali       Sedan         89398         63   
4  AC-100004   60  Female       Tigray         SUV         78475         69   

   AnnualPremium  Deductible  NCD  ...  Claimed  ClaimAmount  TotalPremium  \
0           2346         500   30  ...    False          0.0          2346   
1           2334         500    0  ...     True       9883.0          2334   
2           1697         250   20  ...    False          0.0          1697   
3           2370         500   20  ...     True      12134.0          2370   
4           2582         500    0  ...    False          0.0          2582   

   TotalClaims                 CoverType AutoMake  Vehic

In [3]:
df["HasClaim"] = df["TotalClaims"].apply(lambda x: 1 if x > 0 else 0)

print(df["HasClaim"].value_counts())

HasClaim
0    8465
1    1535
Name: count, dtype: int64


# Hypothesis Testing

The objective of this analysis is to statistically evaluate whether risk differs across customer and geographic segments.

The following hypotheses are tested:

1. Risk differences across provinces
2. Risk differences across zip codes
3. Margin differences across zip codes
4. Risk differences between men and women

Statistical significance is evaluated using a threshold of:

α = 0.05

In [4]:
province_claims = df.groupby("Province")["TotalClaims"].mean()

print(province_claims.sort_values(ascending=False))

Province
Somali         1542.730574
Oromia         1333.221995
Addis Ababa    1304.516400
Tigray         1302.407960
Amhara         1177.531266
Name: TotalClaims, dtype: float64


In [5]:
group_a = df[df["Province"] == "Gauteng"]["TotalClaims"]

group_b = df[df["Province"] == "Western Cape"]["TotalClaims"]

stat, p = ht.run_ttest(group_a, group_b)

print("T-Statistic:", stat)
print("P-Value:", p)

print(ht.interpret_result(p))

T-Statistic: nan
P-Value: nan
Fail to Reject H0


q:\insurance-risk-analytics\insurance-risk-analytics\src\hypothesis_tests.py:40: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  stat, p_value = ttest_ind(


## Business Interpretation

If the null hypothesis is rejected, this indicates that claim behavior differs significantly between provinces.

This suggests that province-specific pricing adjustments may improve underwriting accuracy and profitability.

In [6]:
male_claims = df[df["Gender"] == "Male"]["TotalClaims"]

female_claims = df[df["Gender"] == "Female"]["TotalClaims"]

stat, p = ht.run_ttest(male_claims, female_claims)

print("T-Statistic:", stat)
print("P-Value:", p)

print(ht.interpret_result(p))

T-Statistic: -0.054709164724640366
P-Value: 0.956371266680949
Fail to Reject H0


## Gender-Based Risk Analysis

This test evaluates whether claim amounts differ significantly between male and female policyholders.

If significant differences exist, ACIS may consider gender-sensitive risk segmentation strategies.

In [11]:
df = ht.calculate_margin(df)

zip_margin = df.groupby("ZipCode")["Margin"].mean()

print(zip_margin.head())

ZipCode
10001    1049.215493
10002    1087.163934
10003    1526.428571
10004    1298.916780
10005     990.325959
Name: Margin, dtype: float64


In [12]:
zip_a = df[df["ZipCode"] == 2000]["Margin"]

zip_b = df[df["ZipCode"] == 8000]["Margin"]

stat, p = ht.run_ttest(zip_a, zip_b)

print("T-Statistic:", stat)
print("P-Value:", p)

print(ht.interpret_result(p))

T-Statistic: nan
P-Value: nan
Fail to Reject H0


q:\insurance-risk-analytics\insurance-risk-analytics\src\hypothesis_tests.py:40: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  stat, p_value = ttest_ind(


In [9]:
results = pd.DataFrame({
    "Hypothesis": [
        "Province Risk Difference",
        "Gender Risk Difference",
        "Zip Code Margin Difference"
    ],

    "Test Used": [
        "T-Test",
        "T-Test",
        "T-Test"
    ],

    "Decision": [
        "Reject/Fail",
        "Reject/Fail",
        "Reject/Fail"
    ]
})

results

,Hypothesis,Test Used,Decision
0,Province Risk Difference,T-Test,Reject/Fail
1,Gender Risk Difference,T-Test,Reject/Fail
2,Zip Code Margin Difference,T-Test,Reject/Fail


# Conclusion

The hypothesis testing phase provides statistical evidence regarding geographic and demographic risk variation.

These findings support the development of more targeted pricing strategies and improved market segmentation for ACIS.